### Conversie taaksjabloon (word -> json)

In [1]:
from ask_delphi_api.convert_taaksjabloon import read_dir, convert_doc_to_json
import pprint

paths = read_dir('/Users/baasa03/Digicoach/taaksjabloon_demo')
json_coaches = []

for path in paths:
    try:
        print("\n\n")
        json_coach = convert_doc_to_json(path)
        json_coaches.append(json_coach)
    except ValueError as e:
        print(e)

LIC - Taaksjablonen Sanering v1.0.docx



retrieved 9 tables from doc /Users/baasa03/Digicoach/taaksjabloon_demo/LIC - Taaksjablonen Sanering v1.0.docx
Found title: Sanering (LIC)
Found tag: Directie CAP
Found tag: Keten Inning en betalingsverkeer
Found step: Open postbak in WAB
Found step: Selecteer alleen kwijtscheldings / saneringsverzoeken
Found step: In behandeling nemen poststuk saneringen
Found step: Controleer taken op je naam
Found task: Selecteer Zaak
Found step: Check op vermogensbestanddelen
Found step: Check op mogelijkheid aansprakelijk stellen
Found step: Check op teruggaven en goeder trouw
Found step: Check op ambtshalve aanslagen
Found step: Vastleggen acties
Found task: Quick scan verzoek
Found step: Controleer competentie
Found step: Controleer de datum van binnenkomst
Found step: Check op benodigde stukken
Found step: Check op onbehandelde zaken
Found step: Controleer volledigheid verzoek
Found step: Vastleggen brief of actie in INL
Found step: Acties bij verzoek is

In [2]:
# print(json_coaches[0])
# topic_id_list = []

### Naamgeving digicoach

In [3]:
import json
from ask_delphi_api.import_digicoach import Import
import datetime

json_digicoach = json.loads(json_coaches[0])
naam_digicoach = f"Digicoach {json_digicoach['name']} {datetime.datetime.now()}"
tags= json_digicoach['tags']
name_voorgedefinieerde_zoekopdracht = ""
for tag in tags:
    if tag["type"] == "Keten" or tag["type"] == "Middel":
        name_voorgedefinieerde_zoekopdracht = tag["values"][0]

print(naam_digicoach)
print(name_voorgedefinieerde_zoekopdracht)

AskDelphi Client loaded.
Digicoach Sanering (LIC) 2026-02-23 11:29:38.781669
Inning en betalingsverkeer


### Creatie voorgedefinieerde_zoekopdracht

In [4]:
from ask_delphi_api.import_digicoach import Import

topic_id_list = []

import_service = Import()
topic_id_predefined_search, topic_version_id_predefined_search = import_service.create_voorgedefinieerde_zoekopdracht_topic(name_voorgedefinieerde_zoekopdracht)
topic_id_list.append(topic_id_predefined_search)

Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!
Created Voorgedefinieerde zoekopdracht topic : 6e495f1d-0746-4e33-b762-52f97b7fc45e


### Creatie Digicoach

In [5]:
topic_id_digicoach, topic_version_id_digicoach = import_service.create_digicoach(naam_digicoach, topic_id_predefined_search, topic_version_id_predefined_search)
topic_id_list.append(topic_id_digicoach)

Created Digicoach topic : fade943e-4166-4ed4-8549-9e38f9767d1d


### Tag Digicoach

In [6]:
# Voeg tags toe
for tag in tags:
    # import_service.add_tag(topic_id_digicoach, topic_version_id_digicoach, tag["values"][0])
    import_service.add_tag(topic_id_digicoach, topic_version_id_digicoach, "interactie")
    # import_service.add_tag(topic_id_digicoach, topic_version_id_digicoach, "CAP")

### Helper functie tbv bronverwijzing

In [7]:
import re

def is_alleen_url(tekst):    
    patroon = r"""^        
        (https?:\/\/)?                 # optioneel http/https        
        ([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,} # domeinnaam        
        (\/\S*)?                       # optioneel pad    
    $"""    
    return re.match(patroon, tekst.strip(), re.VERBOSE) is not None

### Creatie taken, stappen en publiceren

In [8]:
# Haal takenlijst uit json_digicoach
taken = json_digicoach["tasks"]
pprint.pp(taken[0])

# Bronnenlijst uit json_digicoach toevoegen
sources = json_digicoach["sources"]

# Voor elke taak, maak topic
for taak in taken:
    # Maak topic met naam taak uit json
    topic_id_task, topic_version_id_task = import_service.create_task(taak["name"], topic_id_digicoach, topic_version_id_digicoach)
    topic_id_list.append(topic_id_task)

    # Voeg beschrijving als content toe
    import_service.add_content_to_topic(topic_id_task, topic_version_id_task, taak['description'])

    # Voeg handleiding en instructies toe
    for source in sources:
        if source["type"] == "Handleidingen en instructies":
            if is_alleen_url(source["link"]):
                # pprint.pp(source)
                topic_id_source, topic_version_id_source = import_service.add_source(topic_id_task, topic_version_id_task, source)
                import_service._get_topic().checkin(topic_id_source)
                topic_id_list.append(topic_id_source)

    # Haal stappenlijst uit taak
    steps = taak["steps"]
    # pprint.pp(steps[0])

    for step in steps:
    # Maak topic met naam step uit json
        topic_id_step, topic_version_id_step = import_service.create_step(step["name"], topic_id_task, topic_version_id_task)
        topic_id_list.append(topic_id_step)

        # Voeg beschrijving als content toe
        import_service.add_content_to_topic(topic_id_step, topic_version_id_step, step['description'])

        import_service._get_topic().checkin(topic_id_step)

    # Inchecken taak
    import_service._get_topic().checkin(topic_id_task)

import_service._get_topic().checkin(topic_id_digicoach)
import_service._get_topic().checkin(topic_id_predefined_search)

for topic_id in topic_id_list:
    import_service.publiceer(topic_id)


{'name': 'Selecteer Zaak',
 'description': 'In deze taak start je je werkzaamheden in WAB post op om de '
                'saneringsverzoeken te behandelen.',
 'steps': [{'name': 'Open postbak in WAB',
            'description': 'Ga naar de WAB-Post omgeving via de link '
                           'https://wab.belastingdienst.nl/ProcessPortal'},
           {'name': 'Selecteer alleen kwijtscheldings / saneringsverzoeken',
            'description': 'Links in het grijze gedeelte druk je op de saved '
                           'search ‘’Taskforce KWT/SAN’’, ‘’Taskforce KWT/SAN '
                           'reactie ontvangen’’ en ‘’Taskforce KWT/SAN Reactie '
                           'termijn afgelopen’’.  Op deze manier verschijnen '
                           'de saneringsverzoeken in de werkomgeving.\n'
                           '\n'
                           'Je kunt deze saved search favoriet maken door op '
                           'het vergrootglas vóór de saved search te kl